# Module 4 Lab: PySpark & Lakehouse Interactive Deep Dive
Welcome! In this hands-on lab, we explore distributed computing with **PySpark** and high-performance in-process analytics with **DuckDB**, reading and writing to our mock Cloud S3 Data Lake (**MinIO**).

In [ ]:
import os
import duckdb
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum, avg

print("Libraries successfully imported!")

## 1. Initializing PySpark Session with MinIO S3A Support

In [ ]:
spark = SparkSession.builder \
    .appName("DE_Interactive_Lab") \
    .master("local[*]") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Version:", spark.version)

## 2. Generate Synthetic Distributed DataFrame

In [ ]:
data = [
    (1, "Laptop Pro", "Electronics", 1299.99, "2026-10-01"),
    (2, "Wireless Earbuds", "Audio", 89.50, "2026-10-01"),
    (3, "Desk Chair", "Furniture", 249.00, "2026-10-02"),
    (4, "USB-C Hub", "Electronics", 45.00, "2026-10-02"),
    (5, "Smart Watch", "Electronics", 199.99, "2026-10-03"),
    (6, "Studio Monitor", "Audio", 350.00, "2026-10-03")
]
columns = ["item_id", "name", "category", "price", "order_date"]

df = spark.createDataFrame(data, columns)
df.show()

## 3. Aggregations: Narrow vs. Wide Dependencies (Catalyst Optimizer in action)

In [ ]:
category_kpis = df.groupBy("category").agg(
    count("item_id").alias("total_items"),
    _sum("price").alias("total_revenue"),
    avg("price").alias("avg_price")
)

category_kpis.explain(True)
category_kpis.show()

## 4. Querying Lakehouse Parquet with DuckDB (Zero-Copy OLAP)

In [ ]:
con = duckdb.connect()
con.execute("""
    INSTALL httpfs;
    LOAD httpfs;
    SET s3_endpoint='minio:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")
print("DuckDB S3 HTTPFS Extension Loaded!")